# Bayer Leverkusen 2023/24 — Passing EDA

## Purpose

Explore how Bayer Leverkusen moved the ball during the 2023/24 Bundesliga season.

Initial questions include:

- How much did Leverkusen pass?
- How successful were those passes?
- Which players passed and received the ball most often?
- Where on the pitch did passes originate and end?
- How did passing vary by match and opponent?
- Which passes progressed possession?
- Which player combinations formed the strongest passing relationships?

In [2]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_matches, load_events

TEAM_NAME = "Bayer Leverkusen"

In [4]:
matches = load_matches()

len(matches)

34

In [5]:
season_pass_frames = []

for match in matches:
    match_id = match["match_id"]

    events = load_events(match_id)

    events_df = pd.json_normalize(
        events,
        sep="_"
    )

    match_passes = events_df[
        (events_df["type_name"] == "Pass")
        & (events_df["team_name"] == TEAM_NAME)
    ].copy()

    match_passes["match_id"] = match_id
    match_passes["match_date"] = pd.to_datetime(
        match["match_date"]
    )

    if match["home_team"]["home_team_name"] == TEAM_NAME:
        opponent = match["away_team"]["away_team_name"]
        venue = "Home"
    else:
        opponent = match["home_team"]["home_team_name"]
        venue = "Away"

    match_passes["opponent"] = opponent
    match_passes["venue"] = venue

    season_pass_frames.append(match_passes)

In [10]:
len(season_pass_frames)

34

In [7]:
type(season_pass_frames[0])

pandas.DataFrame

In [8]:
season_passes_df = pd.concat(
    season_pass_frames,
    ignore_index=True
)

In [9]:
season_passes_df.shape

(24244, 142)

In [12]:
season_passes_df["match_id"].nunique()

34

In [13]:
season_passes_df["completed"] = (
    season_passes_df["pass_outcome_name"].isna()
)

In [14]:
season_passes_df["start_x"] = (
    season_passes_df["location"].str[0]
)

season_passes_df["start_y"] = (
    season_passes_df["location"].str[1]
)

season_passes_df["end_x"] = (
    season_passes_df["pass_end_location"].str[0]
)

season_passes_df["end_y"] = (
    season_passes_df["pass_end_location"].str[1]
)

In [15]:
print("Passes:", len(season_passes_df))
print("Matches:", season_passes_df["match_id"].nunique())
print("Players:", season_passes_df["player_name"].nunique())

Passes: 24244
Matches: 34
Players: 24


In [16]:
season_passes_df["completed"].value_counts()

completed
True     21351
False     2893
Name: count, dtype: int64

In [18]:
season_completion_pct = (
    season_passes_df["completed"].mean() * 100
)

round(season_completion_pct, 2)

np.float64(88.07)

In [19]:
season_passes_df.groupby("match_id").size().describe()

count      34.000000
mean      713.058824
std       134.759481
min       441.000000
25%       637.500000
50%       732.000000
75%       785.750000
max      1002.000000
dtype: float64